<a href="https://colab.research.google.com/github/rist-kobe/HPC-Programming/blob/main/Tuning/sample_code/05_mattp/05_mattp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

Install GNU Fortran and NVIDIA HPC SDK (optional for C-only examples; can take ~30 min)

In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y build-essential gfortran curl gnupg

# Optional: install NVIDIA HPC SDK (large download, not needed for C-only examples).
# Uncomment the following lines to install it.
#!curl -fsSL https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg
#!echo 'deb [signed-by=/usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /' | sudo tee /etc/apt/sources.list.d/nvhpc.list
#!sudo apt-get update -y
#!sudo apt-get install -y nvhpc-22-7-cuda-multi

import glob, os
nvhpc_bins = sorted(glob.glob('/opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin'), key=lambda p: tuple(int(x) for x in p.split('/Linux_x86_64/')[1].split('/')[0].replace('-', '.').split('.')), reverse=True)
if nvhpc_bins:
    nvhpc_bin = nvhpc_bins[0]
    current_path = os.environ.get('PATH', '')
    if nvhpc_bin not in current_path.split(':'):
        os.environ['PATH'] = nvhpc_bin + (':' + current_path if current_path else '')
    print('Using NVIDIA HPC SDK:', nvhpc_bin)
else:
    print('NVIDIA HPC SDK not found (optional; not needed for C-only examples).')


Clone the repository and change to the `05_mattp` directory.

In [ ]:
%cd /content
!rm -rf HPC-Programming
!git clone https://github.com/rist-kobe/HPC-Programming.git
%cd HPC-Programming/Tuning/sample_code/05_mattp
!ls


# Loop blocking: transpose a matrix 
* Author:   Yukihiro Ota (yota@rist.or.jp)
* Last update: 23rd Jan., 2024 

## Instruction: Compile
1. Source code is stored in `src/`. Choose either fortran or c.
2. Change directory

In [ ]:
!cd src/c # On c


3. Make

In [ ]:
!make


The code is successfully compiled by GNU compiler (8.5.0 and 12.2.0) on x86-64 systems. 

## Instruction: Run and do a performance analysis
1. Sample files are stored in `tests/`. Choose either fortran or c.
2. Change directory

In [ ]:
!cd tests/c # On c


3. Run a job script, `run.sh`.

In [ ]:
# One example
!bash run.sh
# Another example
!chmod 755 run.sh
!./run.sh


4. Output is summarized in `outlist`. Compare the runtime between **Matrix transpose (standard)** and **Matrix transpose (blocking)**. 

## Exercise
1. Examine the sizes of L1 and L2 (and last-level) cache in your machine. In Linux, you can obtain this information typing `cat /proc/cpuinfo`. The number in `cache size` would be an answer. When you have multi-core machines, the last-level (e.g. L3) cache is typically shared with several cores.  
2. Run jobs changing matrix row size with a fixed small block size. Look at `Req. memory` in `outlist` Carefully observe behaviors when `Req. memory` is set around the size of a cache (per core). 
3. Set matrix row size so that `Req. memory` is far beyond a cache size. Then, measure the runtime changing block size. 

## Advanced topics
1. Examine the size of cache line in your machine. In Linux, use of `getconf` is straightforward. Also, `cpuid` is quite useful. How many array elements are included in one cache line if an element of an array is double precision?

In [ ]:
!getconf LEVEL2_CACHE_LINESIZE
!getconf LEVEL1_DCACHE_LINESIZE
!getconf -a |grep -E "CACHE"


2. If you have a profiler to obtain hardware events, measure memory access throughput (or memory bandwidth), rather than runtime.
3. Try to measure memory latency using [LMbench](https://lmbench.sourceforge.net/). 
  * `LMbench` requires `libtirpc`. If the library and the related header file are absent in your machine, you need to obtain it. On RHEL8, the corresponding library is involved in `libtirpc-devel`.